In [ ]:
import pickle
import socket
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display

np.random.seed(0)
torch.manual_seed(0)
import sys

sys.path.insert(0, str(Path.cwd().parent))

from fig_utils.transformed_rnn import transformed_rnn
from vi_rnn.datasets import SWM_dataset_multi
from vi_rnn.data_utils import make_all_trials
from vi_rnn.saving import CPU_Unpickler, load_model


from fig_utils.decoding import (
    eval_gen_decoder,
    get_data_for_decoding,
    print_acc_summary,
    train_shared_decoder,
)
from fig_utils.plots import plot_basis_2d_subspaces, position_slice_at_time


%matplotlib inline

In [ ]:
hostname = socket.gethostname()
print("hostname:", hostname)

if hostname == "MatthijsDesktop":
    out_dir = Path("/home/matthijs/swm_rnn/final_models/macaque")
    path = "/home/matthijs/swm_rnn/data/"

else:
    out_dir = Path("/Users/matthijs/swm_rnn/final_models/macaque")
    path = str(Path.cwd().parent / "data") + "/"


model_dirs = [
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_28_T_05_34_01",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_05_01_T_22_03_56",
    "SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_18_03_12",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_17_02_13",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_30_36",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_29_T_16_26_38",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_27_25",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_30_22",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_20_35",
    "models_SWM_low_rank_one_to_one_dim_z_64_date_2026_04_30_T_20_13_05",
]

In [ ]:
# load the basis dataframe
df_basii = pickle.load(open("../data/processed/df_basii.pkl", "rb"))

In [ ]:
# --- controls ---

# --- task ---
n_pos = 3
n_stim = 6

# --- data ---
n_trials_per_ses = 150
n_trials_per_ses_eval = 50
n_duplications = 10

# --- windows ---
t_plot = 65
bins_before = 4
bins_after = 1

# --- decoder ---
C = 1
fit_intercept = False
max_iter = 1000

# --- style ---
generate_plots = False
cmap = mpl.colors.ListedColormap(sns.color_palette("husl", n_colors=n_stim))

# --- run ---
n_pcs_time = 2
run = False

In [ ]:
task_params_file = str(out_dir) + "/" + model_dirs[0] + "_task_params.pkl"
with open(task_params_file, "rb") as f:
    task_params = CPU_Unpickler(f).load()
bin_size = task_params["bin_size"]

u, _, labels, delay_ends = make_all_trials(
    task_params,
    dur=5.5,
    n_stim=n_stim,
    n_pos=n_pos,
    cue_dur=None,
    bin_size=task_params["bin_size"],
    interval_dur="mean",
    delay_dur="mean",
)

u_gen = np.concatenate([u] * n_duplications, axis=0)
labels_gen = np.concatenate([labels] * n_duplications, axis=0)
delay_ends_gen = np.concatenate([delay_ends] * n_duplications, axis=0)

In [ ]:
if run:
    rows = []

    for i, model_dir in enumerate(model_dirs[:10]):
        model_dir = out_dir / Path(model_dir)
        name = model_dir.name

        row_b = df_basii.loc[df_basii["name"] == name]
        if row_b.empty:
            print(f"skip {name}: not in df_basii")
            continue

        print("\n ------------- \n")
        print("model name:", name)
        print(f"Processing model {i + 1} of {len(model_dirs[:10])}")

        vae, training_params, task_params = load_model(
            str(model_dir), load_encoder=False, backward_compat=False
        )
        task_params["path"] = path
        task = SWM_dataset_multi(task_params)

        A_comb_np = row_b["A_comb_np"].values[0]
        b_comb_np = row_b["b_comb_np"].values[0]
        rnn_orth = transformed_rnn(vae, A_comb_np, b_comb_np)
        print("macaque:", task_params["sessions"][0][5:10])

        z_gen = rnn_orth.simulate(u_gen)
        if generate_plots:
            z_by_pos = [
                position_slice_at_time(z_gen, t_plot, p, n_pcs_time)
                for p in range(n_pos)
            ]
            c_by_pos = [labels_gen[:, p] for p in range(n_pos)]
            plot_basis_2d_subspaces(
                z_by_pos, c_by_pos, cmap=cmap, n_stim=n_stim, n_pos=n_pos
            )

        k_model = training_params["k"]
        (
            mean_Qzs,
            labels_dec,
            mean_Qzs_test,
            labels_test,
            mean_response_onsets,
        ) = get_data_for_decoding(
            vae,
            task,
            A_comb_np,
            b_comb_np,
            n_trials_per_ses,
            n_trials_per_ses_eval,
            bins_before,
            bins_after,
            k_model,
        )
        print(mean_response_onsets)
        shared_model, shared_acc, w_eff, b_eff = train_shared_decoder(
            mean_Qzs,
            labels_dec,
            mean_Qzs_test,
            labels_test,
            C=C,
            max_iter=max_iter,
            fit_intercept=fit_intercept,
        )
        print("Shared decoder accuracy:", shared_acc)

        acc_gen = eval_gen_decoder(
            z_gen,
            labels_gen,
            delay_ends_gen,
            shared_model,
            mean_response_onsets,
            n_pos,
            bins_before,
            bins_after,
        )
        print("Gen decoder accuracy:", acc_gen)

        rows.append(
            {
                "name": name,
                "path": str(model_dir),
                "macaque": task_params["sessions"][0][5:10],
                "w_eff": w_eff,
                "b_eff": b_eff,
                "shared_acc": shared_acc,
                "gen_acc": acc_gen,
                "mean_response_onsets": mean_response_onsets,
            }
        )

In [ ]:
if run:
    df = pd.DataFrame(rows)
    df.to_pickle("../data/processed/df_decoding.pkl")
else:
    df = pd.read_pickle("../data/processed/df_decoding.pkl")

for acc_col in ("shared_acc", "gen_acc"):
    print("=" * 60)
    print(acc_col)
    print_acc_summary(df, "All models", acc_col)
    for species in ("groot", "ocean"):
        print_acc_summary(df[df["macaque"] == species], species, acc_col)